## Import libraries

In [1]:
import sys
from pathlib import Path
import pandas as pd
import csv
from graphs import Neo4jQuery

import scispacy
import spacy
from spacy import displacy

# from pprint import pprint
# from spacy import displacy
from scispacy.linking import EntityLinker
from sentence_generator import get_prompt
from sentence_clusterer import SentenceClusterer
import warnings
warnings.filterwarnings("ignore")

Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib
/home/nadun/hdd/miniconda3/envs/titan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extract entities using SciSpacy

In [2]:
text = """The pathology report describes the specimens labeled A, B, and C, which include wide local excisions and an additional margin as part of the surgical treatment of a female patient with bilateral masses and a preoperative diagnosis of bilateral invasive ductal carcinoma. The report reveals that the right breast excision (Part A) measures 2.4 cm and contains invasive ductal carcinoma, SBR grade 2, mixed with ductal carcinoma in situ, lobular carcinoma in situ, and biopsy site changes. The report also indicates that invasive tumor is present at the superior margins and within 1-mm from medial in parts A and B, respectively. The left breast excision (Part C) measures 4 x 4 x 3.5 cm and contains invasive ductal carcinoma, SBR grade 2, and lobular carcinoma in situ, but with negative surgical resection margins. The report further provides additional information, such as specimen processing, margin status, tumor characteristics, lymph node status, and ER/PR/HER2 results."""

text2 = """A pathology report was conducted on breast tissue from the right upper outer quadrant following a radical mastectomy. The grossly evident lesion was 1.8 x 1.6x1.6cm in size and consisted of infiltrating ductal and lobular carcinoma. The tumor extent, lymph nodes, extracapsular invasion, and margins were not specified. 0/4 lymph nodes were positive for metastasis, and the Nottingham Histologic Score could not be determined due to missing information. No evidence of neo-adjuvant treatment or additional pathologic findings were noted."""

In [3]:
ALLOWED_TUIS = {
    "T047",  # Disease or Syndrome
    "T191",  # Neoplastic Process
    "T046",  # Pathologic Function
    "T184",  # Sign or Symptom (optional)
    "T034",  # Laboratory or Test Result (ER/PR/HER2)
}

In [4]:
# Load Scispacy model
nlp = spacy.load("en_core_sci_lg")
nlp.add_pipe(
    "scispacy_linker",
    config={
        "resolve_abbreviations": True,
        "linker_name": "umls"
    }
)

In [9]:
# Process the report text with scispacy
doc = nlp(text)
print(list(doc.sents))

[The pathology report describes the specimens labeled A, B, and C, which include wide local excisions and an additional margin as part of the surgical treatment of a female patient with bilateral masses and a preoperative diagnosis of bilateral invasive ductal carcinoma., The report reveals that the right breast excision (Part A) measures 2.4 cm and contains invasive ductal carcinoma, SBR grade 2, mixed with ductal carcinoma in situ, lobular carcinoma in situ, and biopsy site changes., The report also indicates that invasive tumor is present at the superior margins and within 1-mm from medial in parts A and B, respectively., The left breast excision (Part C) measures 4 x 4 x 3.5 cm and contains invasive ductal carcinoma, SBR grade 2, and lobular carcinoma in situ, but with negative surgical resection margins., The report further provides additional information, such as specimen processing, margin status, tumor characteristics, lymph node status, and ER/PR/HER2 results.]


In [10]:
print(doc.ents)
linker = nlp.get_pipe("scispacy_linker")

(pathology report, specimens, labeled, B, C, local excisions, margin, surgical treatment, female, patient, bilateral masses, preoperative, diagnosis, bilateral invasive ductal carcinoma, report, right breast excision, measures, invasive ductal carcinoma, SBR, ductal carcinoma, in situ, lobular carcinoma, in situ, biopsy site, changes, report, invasive tumor, superior margins, medial, parts A, B, left breast excision, measures, invasive ductal carcinoma, SBR grade 2, lobular carcinoma, in situ, negative, surgical resection margins, information, specimen processing, margin status, tumor characteristics, lymph node status, ER/PR/HER2, results)


In [11]:
for ent in doc.ents:
    for umls_ent in ent._.kb_ents:
        cui, score = umls_ent
        entity = linker.kb.cui_to_entity[cui]

        if any(tui in ALLOWED_TUIS for tui in entity.types):
            print(ent.text, entity.canonical_name, score)


bilateral invasive ductal carcinoma Invasive Ductal Breast Carcinoma 0.8211829662322998
bilateral invasive ductal carcinoma Bilateral Carcinoma 0.7405000925064087
bilateral invasive ductal carcinoma Invasive Breast Ductal Carcinoma and Invasive Lobular Carcinoma 0.7222058176994324
bilateral invasive ductal carcinoma Invasive Ductal and Lobular Carcinoma 0.7068164348602295
invasive ductal carcinoma Invasive Ductal Breast Carcinoma 0.9750464558601379
invasive ductal carcinoma Invasive Breast Ductal Carcinoma and Invasive Lobular Carcinoma 0.8662934899330139
invasive ductal carcinoma Invasive Ductal and Lobular Carcinoma 0.839253306388855
invasive ductal carcinoma Invasive Carcinoma 0.8383232951164246
invasive ductal carcinoma Infiltrating duct and lobular carcinoma 0.8321796655654907
ductal carcinoma Ductal Carcinoma 0.9876976609230042
ductal carcinoma Ductal Breast Carcinoma 0.9876976609230042
ductal carcinoma Noninfiltrating Intraductal Carcinoma 0.8186325430870056
in situ Carcinoma in

In [12]:
# Collect entities with their scores, filtering by allowed TUIs
seen = set()
filtered_entities = []

for ent in doc.ents:
    for umls_ent in ent._.kb_ents:
        cui, score = umls_ent
        entity = linker.kb.cui_to_entity[cui]
        
        # Filter by allowed TUIs
        if any(tui in ALLOWED_TUIS for tui in entity.types):
            # Create a unique key to identify duplicates
            # Using CUI as the unique identifier
            if cui not in seen:
                seen.add(cui)
                filtered_entities.append({
                    'text': ent.text,
                    'canonical_name': entity.canonical_name,
                    'score': score,
                    'cui': cui
                })

# Sort by score in descending order (highest scores first)
filtered_entities.sort(key=lambda x: x['score'], reverse=True)

# Print sorted results
for item in filtered_entities[:10]:
    print(item['text'], item['canonical_name'], item['score'])

lobular carcinoma Carcinoma, Lobular 0.9919567108154297
ductal carcinoma Ductal Carcinoma 0.9876976609230042
ductal carcinoma Ductal Breast Carcinoma 0.9876976609230042
invasive ductal carcinoma Invasive Carcinoma 0.8383232951164246
invasive ductal carcinoma Infiltrating duct and lobular carcinoma 0.8321796655654907
lobular carcinoma Breast Mixed Ductal and Lobular Carcinoma 0.8230723142623901
results Test Result 0.8216436505317688
bilateral invasive ductal carcinoma Invasive Ductal Breast Carcinoma 0.8211829662322998
lobular carcinoma Lobular carcinoma in situ of breast 0.8210324048995972
ductal carcinoma Noninfiltrating Intraductal Carcinoma 0.8186325430870056


## Set-up UMLS server

In [ ]:
# Neo4j connection configuration
NEO4J_URL = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "11111111"  # Change this to your Neo4j password

# CSV file paths (adjust these paths as needed)
NODES_CSV = "/home/nadun/wd/datasets/UMLS_sql/UMLS_DATA/FYP/KGs/nodes.csv"
RELATIONSHIPS_CSV = "/home/nadun/wd/datasets/UMLS_sql/UMLS_DATA/FYP/KGs/relationships.csv"

In [2]:
# Start neo4j query
query = Neo4jQuery(NEO4J_URL, NEO4J_USER, NEO4J_PASSWORD)

# Starting seed
SEED_CUI = "C0007134"

In [3]:
# Get neighbors of the seed CUI
neighbors = query.get_neighbors_filtered(SEED_CUI)

for item in neighbors:
    print(f"CUI: {item['cui']} - {item['name']} - {item['rela']}")

CUI: C4707179 - Acquired cystic disease associated renal cell carcinoma (disorder) - has_associated_morphology
CUI: C2931352 - Familial renal cell carcinoma - has_associated_morphology
CUI: C1288351 - Malignant neoplasm of primary, secondary, or uncertain origin - has_associated_morphology
CUI: C5547215 - Primary renal cell carcinoma of left kidney - has_associated_morphology
CUI: C5547214 - Primary renal cell carcinoma of right kidney (disorder) - has_associated_morphology
CUI: C4305551 - Renal cell carcinoma of kidney except renal pelvis - has_associated_morphology
CUI: C0279702 - hypernephroma - has_associated_morphology
CUI: C4721698 - metastatic renal cell carcinoma - has_associated_morphology
CUI: C3862515 - renal cell carcinoma, bilateral kidneys - has_associated_morphology
